In [1]:
import datetime
from enum import Enum
import os
from typing import Annotated, List, Optional
from dotenv import load_dotenv
from langchain_core.tools import tool

from langchain.agents.structured_output import AutoStrategy,ToolStrategy 
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, ConfigDict, EmailStr, Field, field_validator, model_validator
from rich import print as rprint

load_dotenv(override=True)
ARK_API_KEY = os.getenv("ARK_API_KEY")
ARK_MODEL_NAME = os.getenv("ARK_MODEL_NAME")
ARK_BASE_URL = os.getenv("ARK_BASE_URL")
model = ChatOpenAI(
    model= ARK_MODEL_NAME,
    api_key= ARK_API_KEY,
    base_url= ARK_BASE_URL
)

from datetime import datetime
from enum import Enum
from typing import List, Optional

from pydantic import BaseModel, Field, ConfigDict


# ============ 枚举 ============
class PriorityLevel(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


# ============ 订单信息 ============
class OrderInfo(BaseModel):
    """订单信息"""
    order_no: str = Field(description="订单号")
    product: str = Field(description="商品名称")
    quantity: int = Field(default=1, description="数量")
    amount: float = Field(default=0.0, description="金额（元）")
    priority: PriorityLevel = Field(default=PriorityLevel.MEDIUM, description="优先级")


# ============ 用户信息（含订单号） ============
class UserInfo(BaseModel):
    """用户信息"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱")
    phone: str = Field(description="用户电话")
    order_no: str = Field(description="关联订单号")

    # 通过订单号额外关联的订单信息（可为空）
    order: Optional[OrderInfo] = Field(default=None, description="关联订单信息")


# ================================================================
# 假数据库
# ================================================================
_FAKE_USERS: dict[str, dict] = {
    "张三": {
        "email": "zhangsan@example.com",
        "phone": "138-0000-1234",
        "order_no": "ORD-2024-88888",
    },
    "李四": {
        "email": "lisi@example.com",
        "phone": "139-1111-2222",
        "order_no": "ORD-2024-99999",
    },
}

_FAKE_ORDERS: dict[str, dict] = {
    "ORD-2024-88888": {
        "product": "机械键盘",
        "quantity": 2,
        "amount": 798.0,
        "priority": PriorityLevel.HIGH,
    },
    "ORD-2024-99999": {
        "product": "显示器支架",
        "quantity": 1,
        "amount": 199.0,
        "priority": PriorityLevel.LOW,
    },
}


from langchain_core.tools import tool


# ================================================================
# 方法1：依据用户名返回用户信息（含订单号）
# ================================================================
@tool(parse_docstring=True)
def get_user_by_name(name: str) -> UserInfo:
    """
    按用户名查询用户信息。

    Args:
        name: 用户名，例如「张三」「李四」。

    Returns:
        该用户的 UserInfo 对象，包含姓名、邮箱、电话和关联订单号。
    """
    raw = _FAKE_USERS.get(name)
    if raw is None:
        raise KeyError(f"未找到用户：{name}")

    return UserInfo(
        name=name,
        email=raw["email"],
        phone=raw["phone"],
        order_no=raw["order_no"],
    )


# ================================================================
# 方法2：依据订单号返回订单信息
# ================================================================
@tool(parse_docstring=True)
def get_order_by_no(order_no: str) -> OrderInfo:
    """
    按订单号查询订单信息。

    Args:
        order_no: 订单号，例如「ORD-2024-88888」。

    Returns:
        该订单的 OrderInfo 对象，包含商品、数量、金额和优先级。
    """
    raw = _FAKE_ORDERS.get(order_no)
    if raw is None:
        raise KeyError(f"未找到订单：{order_no}")

    return OrderInfo(order_no=order_no, **raw)




# ============ 6. 创建 agent ============
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="" 
    "你是智能客服【小智】，专门回答用户关于个人账户的问题" 
    "当用户查询个人账户时" 
    "先查询用户名对应的账户信息是否存在" 
    "再依据账户下的订单号查询对应的账户信息" 
    "依据结果生成格式化用户信息对象"),
    response_format=ToolStrategy(schema=UserInfo),
    tools=[get_user_by_name,get_order_by_no]
)

for chunk in agent.stream({
    "messages": [
        (
            "我叫张三，我想知道我最近买了什么？"
        )
        ]
    },
    stream_mode="values"):
    rprint(chunk)
    rprint("-"*50)



c:\Users\XSH\.conda\envs\langchain1.2\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


{
    'messages': [
        HumanMessage(
            content='我叫张三，我想知道我最近买了什么？',
            additional_kwargs={},
            response_metadata={},
            id='4104fa2d-dd58-4372-9d9c-dfd4195c2b42'
        )
    ]
}

--------------------------------------------------

{
    'messages': [
        HumanMessage(
            content='我叫张三，我想知道我最近买了什么？',
            additional_kwargs={},
            response_metadata={},
            id='4104fa2d-dd58-4372-9d9c-dfd4195c2b42'
        ),
        AIMessage(
            content='您好！我是智能客服小智。请稍等，我先为您查询账户信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 109,
                    'prompt_tokens': 613,
                    'total_tokens': 722,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 77,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052276007775251d478e5bf6426f4e0f08f6c1b200a2a1f1',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e379-1b69-76c3-a83c-e5e3c249455f-0',
            tool_calls=[
                {
                    'name': 'get_user_by_name',
                    'args': {'name': '张三'},
                    'id': 'call_krs5fz13i0v1ydypib10uj4d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 613,
                'output_tokens': 109,
                'total_tokens': 722,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 77}
            }
        )
    ]
}

--------------------------------------------------

...

--------------------------------------------------

{
    'messages': [
        HumanMessage(
            content='我叫张三，我想知道我最近买了什么？',
            additional_kwargs={},
            response_metadata={},
            id='4104fa2d-dd58-4372-9d9c-dfd4195c2b42'
        ),
        AIMessage(
            content='您好！我是智能客服小智。请稍等，我先为您查询账户信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 109,
                    'prompt_tokens': 613,
                    'total_tokens': 722,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 77,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052276007775251d478e5bf6426f4e0f08f6c1b200a2a1f1',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e379-1b69-76c3-a83c-e5e3c249455f-0',
            tool_calls=[
                {
                    'name': 'get_user_by_name',
                    'args': {'name': '张三'},
                    'id': 'call_krs5fz13i0v1ydypib10uj4d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 613,
                'output_tokens': 109,
                'total_tokens': 722,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 77}
            }
        ),
        ToolMessage(
            content="name='张三' email='zhangsan@example.com' phone='138-0000-1234' order_no='ORD-2024-88888' 
order=None",
            name='get_user_by_name',
            id='1383d746-8340-476d-9267-5ebb79392661',
            tool_call_id='call_krs5fz13i0v1ydypib10uj4d'
        ),
        AIMessage(
            content='已查到您的账户信息，接下来为您查询订单详情。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 63,
                    'prompt_tokens': 686,
                    'total_tokens': 749,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 30,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052276269375251d478e5bf6426f4e0f08f6c1b20010fffd',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e379-25dd-75a1-b249-626e51b54047-0',
            tool_calls=[
                {
                    'name': 'get_order_by_no',
                    'args': {'order

--------------------------------------------------

{
    'messages': [
        HumanMessage(
            content='我叫张三，我想知道我最近买了什么？',
            additional_kwargs={},
            response_metadata={},
            id='4104fa2d-dd58-4372-9d9c-dfd4195c2b42'
        ),
        AIMessage(
            content='您好！我是智能客服小智。请稍等，我先为您查询账户信息。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 109,
                    'prompt_tokens': 613,
                    'total_tokens': 722,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 77,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052276007775251d478e5bf6426f4e0f08f6c1b200a2a1f1',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e379-1b69-76c3-a83c-e5e3c249455f-0',
            tool_calls=[
                {
                    'name': 'get_user_by_name',
                    'args': {'name': '张三'},
                    'id': 'call_krs5fz13i0v1ydypib10uj4d',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 613,
                'output_tokens': 109,
                'total_tokens': 722,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 77}
            }
        ),
        ToolMessage(
            content="name='张三' email='zhangsan@example.com' phone='138-0000-1234' order_no='ORD-2024-88888' 
order=None",
            name='get_user_by_name',
            id='1383d746-8340-476d-9267-5ebb79392661',
            tool_call_id='call_krs5fz13i0v1ydypib10uj4d'
        ),
        AIMessage(
            content='已查到您的账户信息，接下来为您查询订单详情。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 63,
                    'prompt_tokens': 686,
                    'total_tokens': 749,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 30,
                        'rejected_prediction_tokens': None,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': None,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'glm-5-3-flash-260828',
                'system_fingerprint': None,
                'id': '02179052276269375251d478e5bf6426f4e0f08f6c1b20010fffd',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a0e379-25dd-75a1-b249-626e51b54047-0',
            tool_calls=[
                {
                    'name': 'get_order_by_no',
                    'args': {'order

--------------------------------------------------

当stream_model =  values 时，每一个步骤执行后都会输出完整的状态请求信息

In [2]:
for chunk in agent.stream({
    "messages": [
        (
            "我叫张三，我想知道我最近买了什么？"
        )
        ]
    },
    stream_mode="updates"):
    rprint(chunk)
    rprint("-"*50)

{
    'model': {
        'messages': [
            AIMessage(
                content='您好，张三！我是智能客服小智，我来帮您查询一下您的账户和订单信息。\n\n首先查询您的账户信息
：',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 196,
                        'prompt_tokens': 613,
                        'total_tokens': 809,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 154,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790522902914534779bc6a83ab3f36a7a90c1e71ba35acf428',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0e37b-495f-72e0-b7aa-4dc7745cc381-0',
                tool_calls=[
                    {
                        'name': 'get_user_by_name',
                        'args': {'name': '张三'},
                        'id': 'call_qr4dkykl4de0kds0gwq3ghty',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 613,
                    'output_tokens': 196,
                    'total_tokens': 809,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 154}
                }
            )
        ]
    }
}

--------------------------------------------------

{
    'tools': {
        'messages': [
            ToolMessage(
                content="name='张三' email='zhangsan@example.com' phone='138-0000-1234' order_no='ORD-2024-88888' 
order=None",
                name='get_user_by_name',
                id='01b909a3-08e5-4412-b12a-52ff161fe5cf',
                tool_call_id='call_qr4dkykl4de0kds0gwq3ghty'
            )
        ]
    }
}

--------------------------------------------------

{
    'model': {
        'messages': [
            AIMessage(
                content='已找到您的账户信息，现在查询您名下订单的详细信息：',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 64,
                        'prompt_tokens': 696,
                        'total_tokens': 760,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 29,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790522905858534779bc6a83ab3f36a7a90c1e71ba3525d68c',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0e37b-551d-7d63-885c-3e8c98faa85e-0',
                tool_calls=[
                    {
                        'name': 'get_order_by_no',
                        'args': {'order_no': 'ORD-2024-88888'},
                        'id': 'call_j1dtp8rj6v7dxjgg2w41flle',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 696,
                    'output_tokens': 64,
                    'total_tokens': 760,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 29}
                }
            )
        ]
    }
}

--------------------------------------------------

{
    'tools': {
        'messages': [
            ToolMessage(
                content="order_no='ORD-2024-88888' product='机械键盘' quantity=2 amount=798.0 
priority=<PriorityLevel.HIGH: 'high'>",
                name='get_order_by_no',
                id='825928b6-a385-4857-bc89-2ece5c7d0cf6',
                tool_call_id='call_j1dtp8rj6v7dxjgg2w41flle'
            )
        ]
    }
}

--------------------------------------------------

{
    'model': {
        'messages': [
            AIMessage(
                content='张三您好！查询完成，这是您的账户信息：\n\n```json\n{\n  "name": "张三",\n  "email": 
"zhangsan@example.com",\n  "phone": "138-0000-1234",\n  "order_no": "ORD-2024-88888",\n  "order": {\n    
"order_no": "ORD-2024-88888",\n    "product": "机械键盘",\n    "quantity": 2,\n    "amount": 798.0,\n    
"priority": "high"\n  }\n}\n```\n\n**您最近的购买记录如下：**\n\n- 🛒 **商品名称**：机械键盘\n- 🔢 **购买数量**：2 
个\n- 💰 **订单金额**：¥798.00\n- 📋 **订单编号**：ORD-2024-88888\n- ⚡ 
**订单优先级**：高\n\n请问还有什么可以帮您的吗？比如查询物流进度或办理售后等。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 240,
                        'prompt_tokens': 771,
                        'total_tokens': 1011,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 31,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790522907928534779bc6a83ab3f36a7a90c1e71ba35601f96',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a0e37b-5d33-7c93-a2f4-78e2ff79518d-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 771,
                    'output_tokens': 240,
                    'total_tokens': 1011,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 31}
                }
            )
        ]
    }
}

--------------------------------------------------

stream_values = updates 这也是默认的指定模型，步骤执行后，只显示增量的内容

In [3]:
for chunk in agent.stream({
    "messages": [
        (
            "我叫张三，我想知道我最近买了什么？"
        )
        ]
    },
    stream_mode="messages"):
    rprint(chunk)
    rprint("-"*50)

(
    AIMessageChunk(
        content='',
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a0e37c-753b-7b91-9fcb-4debec82e623',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'ls_provider': 'openai',
        'ls_model_name': 'glm-5-3-flash-260828',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.2.12', 'langchain-openai': '1.6.2'}
    }
)

--------------------------------------------------

(
    AIMessageChunk(
        content='',
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a0e37c-753b-7b91-9fcb-4debec82e623',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'ls_provider': 'openai',
        'ls_model_name': 'glm-5-3-flash-260828',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.2.12', 'langchain-openai': '1.6.2'}
    }
)

--------------------------------------------------

(
    AIMessageChunk(
        content='',
        additional_kwargs={},
        response_metadata={'model_provider': 'openai'},
        id='lc_run--01a0e37c-753b-7b91-9fcb-4debec82e623',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[]
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 1,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'checkpoint_ns': 'model:83b058b5-c2ef-5669-9145-cc052122f6f4',
        'ls_provider': 'openai',
        'ls_model_name': 'glm-5-3-flash-260828',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.2.12', 'langchain-openai': '1.6.2'}
    }
)

...

--------------------------------------------------

(
    AIMessageChunk(
        content='',
        additional_kwargs={},
        response_metadata={},
        id='lc_run--01a0e37c-b4f6-7901-b1a4-da2afc058bdf',
        tool_calls=[],
        invalid_tool_calls=[],
        tool_call_chunks=[],
        chunk_position='last'
    ),
    {
        'ls_integration': 'langchain_chat_model',
        'langgraph_step': 9,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:3458c0ca-7043-465d-7278-74a31dc4e643',
        'checkpoint_ns': 'model:3458c0ca-7043-465d-7278-74a31dc4e643',
        'ls_provider': 'openai',
        'ls_model_name': 'glm-5-3-flash-260828',
        'ls_model_type': 'chat',
        'ls_temperature': None,
        'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.2.12', 'langchain-openai': '1.6.2'}
    }
)

--------------------------------------------------

(
    ToolMessage(
        content="Returning structured response: name='张三' email='zhangsan@example.com' phone='138-0000-1234' 
order_no='ORD-2024-88888' order=None",
        name='UserInfo',
        id='557f00f5-b408-4b30-b458-6e453e3cc3c1',
        tool_call_id='call_xiyoocj9uy0qytnimqk3jdi3'
    ),
    {
        'ls_integration': 'langgraph',
        'langgraph_step': 9,
        'langgraph_node': 'model',
        'langgraph_triggers': ('branch:to:model',),
        'langgraph_path': ('__pregel_pull', 'model'),
        'langgraph_checkpoint_ns': 'model:3458c0ca-7043-465d-7278-74a31dc4e643',
        'LANGSMITH_ENDPOINT': 'https://api.smith.langchain.com',
        'LANGSMITH_PROJECT': 'langchainTest',
        'LANGSMITH_TRACING': 'true',
        'revision_id': 'ebb1529'
    }
)

--------------------------------------------------

stream = message,每一个字段的更新都进行显示

太费token了TAT

In [10]:
for chunk in agent.stream({
    "messages": [
        (
            "我叫张三，我想知道我最近买了什么？"
        )
        ]
    },
    stream_mode="tasks"):
    rprint(chunk)
    rprint("-"*50)

{
    'id': '8b89a800-687e-d53f-2c3a-6c166d2c4e24',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='我叫张三，我想知道我最近买了什么？',
                additional_kwargs={},
                response_metadata={},
                id='49ad8208-99e9-4342-a9b6-3b9a645840ec'
            )
        ]
    },
    'triggers': ('branch:to:model',)
}

--------------------------------------------------

{
    'id': '8b89a800-687e-d53f-2c3a-6c166d2c4e24',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='您好！我是智能客服小智。让我先为您查询账户信息。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 175,
                        'prompt_tokens': 613,
                        'total_tokens': 788,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 146,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790523652305b4a50cd25f47d4bb7d101bbb29abc519bc97a0',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0e386-b8a7-71c0-9a75-e0cc04640594-0',
                tool_calls=[
                    {
                        'name': 'get_user_by_name',
                        'args': {'name': '张三'},
                        'id': 'call_svmykyrnts7crddyxh088qih',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 613,
                    'output_tokens': 175,
                    'total_tokens': 788,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 146}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------

{
    'id': 'cbb26791-0802-2ffd-fec4-89ed7891e65e',
    'name': 'tools',
    'input': {
        '__type': 'tool_call_with_context',
        'tool_call': {
            'name': 'get_user_by_name',
            'args': {'name': '张三'},
            'id': 'call_svmykyrnts7crddyxh088qih',
            'type': 'tool_call'
        },
        'state': {
            'messages': [
                HumanMessage(
                    content='我叫张三，我想知道我最近买了什么？',
                    additional_kwargs={},
                    response_metadata={},
                    id='49ad8208-99e9-4342-a9b6-3b9a645840ec'
                ),
                AIMessage(
                    content='您好！我是智能客服小智。让我先为您查询账户信息。',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 175,
                            'prompt_tokens': 613,
                            'total_tokens': 788,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 146,
                                'rejected_prediction_tokens': None,
                                'text_tokens': None
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'image_tokens': None,
                                'text_tokens': None
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5-3-flash-260828',
                        'system_fingerprint': None,
                        'id': '021790523652305b4a50cd25f47d4bb7d101bbb29abc519bc97a0',
                        'service_tier': 'default',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0e386-b8a7-71c0-9a75-e0cc04640594-0',
                    tool_calls=[
                        {
                            'name': 'get_user_by_name',
                            'args': {'name': '张三'},
                            'id': 'call_svmykyrnts7crddyxh088qih',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 613,
                        'output_tokens': 175,
                        'total_tokens': 788,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 146}
                    }
                )
            ]
        }
    },
    'triggers': ('__pregel_push',)
}

...

--------------------------------------------------

{
    'id': 'ef1adfe7-875e-4840-ad0f-ead71fb9fcd8',
    'name': 'model',
    'input': {
        'messages': [
            HumanMessage(
                content='我叫张三，我想知道我最近买了什么？',
                additional_kwargs={},
                response_metadata={},
                id='49ad8208-99e9-4342-a9b6-3b9a645840ec'
            ),
            AIMessage(
                content='您好！我是智能客服小智。让我先为您查询账户信息。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 175,
                        'prompt_tokens': 613,
                        'total_tokens': 788,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 146,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790523652305b4a50cd25f47d4bb7d101bbb29abc519bc97a0',
                    'service_tier': 'default',
                    'finish_reason': 'tool_calls',
                    'logprobs': None
                },
                id='lc_run--01a0e386-b8a7-71c0-9a75-e0cc04640594-0',
                tool_calls=[
                    {
                        'name': 'get_user_by_name',
                        'args': {'name': '张三'},
                        'id': 'call_svmykyrnts7crddyxh088qih',
                        'type': 'tool_call'
                    }
                ],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 613,
                    'output_tokens': 175,
                    'total_tokens': 788,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 146}
                }
            ),
            ToolMessage(
                content="name='张三' email='zhangsan@example.com' phone='138-0000-1234' order_no='ORD-2024-88888' 
order=None",
                name='get_user_by_name',
                id='25a223e5-42c3-4002-a1f3-ee5051d7cd25',
                tool_call_id='call_svmykyrnts7crddyxh088qih'
            ),
            AIMessage(
                content='已找到您的账户信息，现在为您查询关联订单的详情。',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 61,
                        'prompt_tokens': 683,
                        'total_tokens': 744,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 27,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                

--------------------------------------------------

{
    'id': 'ef1adfe7-875e-4840-ad0f-ead71fb9fcd8',
    'name': 'model',
    'error': None,
    'result': {
        'messages': [
            AIMessage(
                content='查询完成！以下是您的格式化用户信息：\n\n## 👤 用户信息\n\n| 项目 | 内容 
|\n|------|------|\n| **姓名** | 张三 |\n| **邮箱** | zhangsan@example.com |\n| **电话** | 138-0000-1234 |\n| 
**关联订单号** | ORD-2024-88888 |\n\n## 📦 订单详情\n\n| 项目 | 内容 |\n|------|------|\n| **商品名称** | 机械键盘 
|\n| **数量** | 2 |\n| **金额** | 798.00 元 |\n| **优先级** | 高 |\n\n```json\n{\n  "name": "张三",\n  "email": 
"zhangsan@example.com",\n  "phone": "138-0000-1234",\n  "order_no": "ORD-2024-88888",\n  "order": {\n    
"order_no": "ORD-2024-88888",\n    "product": "机械键盘",\n    "quantity": 2,\n    "amount": 798.0,\n    
"priority": "high"\n  }\n}\n```\n\n**总结**：张三您好，您最近购买的是 **机械键盘 × 2**，订单金额为 **798 
元**。如果您还有其他问题，比如查询订单物流状态或售后问题，随时告诉我哦！😊',
                additional_kwargs={'refusal': None},
                response_metadata={
                    'token_usage': {
                        'completion_tokens': 305,
                        'prompt_tokens': 757,
                        'total_tokens': 1062,
                        'completion_tokens_details': {
                            'accepted_prediction_tokens': None,
                            'audio_tokens': None,
                            'reasoning_tokens': 21,
                            'rejected_prediction_tokens': None,
                            'text_tokens': None
                        },
                        'prompt_tokens_details': {
                            'audio_tokens': None,
                            'cache_write_tokens': None,
                            'cached_tokens': 0,
                            'image_tokens': None,
                            'text_tokens': None
                        }
                    },
                    'model_provider': 'openai',
                    'model_name': 'glm-5-3-flash-260828',
                    'system_fingerprint': None,
                    'id': '021790523658082b4a50cd25f47d4bb7d101bbb29abc519e01785',
                    'service_tier': 'default',
                    'finish_reason': 'stop',
                    'logprobs': None
                },
                id='lc_run--01a0e386-cf7b-7d62-919f-06b307700583-0',
                tool_calls=[],
                invalid_tool_calls=[],
                usage_metadata={
                    'input_tokens': 757,
                    'output_tokens': 305,
                    'total_tokens': 1062,
                    'input_token_details': {'cache_read': 0},
                    'output_token_details': {'reasoning': 21}
                }
            )
        ]
    },
    'interrupts': []
}

--------------------------------------------------

stream_mode="tasks" 的输出模式和前面几种都不同。它不会直接给你消息内容，而是发出任务的启动和结束事件，用来监控每个节点什么时候开始跑、什么时候跑完、有没有报错。

In [11]:
for chunk in agent.stream({
    "messages": [
        (
            "我叫张三，我想知道我最近买了什么？"
        )
        ]
    },
    stream_mode="debug"):
    rprint(chunk)
    rprint("-"*50)

{
    'step': 1,
    'timestamp': '2026-09-27T15:42:45.941079+00:00',
    'type': 'task',
    'payload': {
        'id': 'e6a401bb-5646-343b-f392-55e738751595',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='我叫张三，我想知道我最近买了什么？',
                    additional_kwargs={},
                    response_metadata={},
                    id='7699923e-eeef-4179-99b5-9e5f62353dec'
                )
            ]
        },
        'triggers': ('branch:to:model',)
    }
}

--------------------------------------------------

{
    'step': 1,
    'timestamp': '2026-09-27T15:42:48.644190+00:00',
    'type': 'task_result',
    'payload': {
        'id': 'e6a401bb-5646-343b-f392-55e738751595',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='您好，我是智能客服【小智】！我先为您查询账户信息~',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 149,
                            'prompt_tokens': 613,
                            'total_tokens': 762,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 119,
                                'rejected_prediction_tokens': None,
                                'text_tokens': None
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'image_tokens': None,
                                'text_tokens': None
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5-3-flash-260828',
                        'system_fingerprint': None,
                        'id': '021790523765974923718d11d313173fa258a554449eb87273a3c',
                        'service_tier': 'default',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0e388-74b9-7351-96cd-1fc1ab298e0c-0',
                    tool_calls=[
                        {
                            'name': 'get_user_by_name',
                            'args': {'name': '张三'},
                            'id': 'call_ro4xglizn6bblv8shb5fg0mj',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 613,
                        'output_tokens': 149,
                        'total_tokens': 762,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 119}
                    }
                )
            ]
        },
        'interrupts': []
    }
}

--------------------------------------------------

{
    'step': 2,
    'timestamp': '2026-09-27T15:42:48.644362+00:00',
    'type': 'task',
    'payload': {
        'id': '835c398b-28f0-d0a1-47b0-f582cee6138b',
        'name': 'tools',
        'input': {
            '__type': 'tool_call_with_context',
            'tool_call': {
                'name': 'get_user_by_name',
                'args': {'name': '张三'},
                'id': 'call_ro4xglizn6bblv8shb5fg0mj',
                'type': 'tool_call'
            },
            'state': {
                'messages': [
                    HumanMessage(
                        content='我叫张三，我想知道我最近买了什么？',
                        additional_kwargs={},
                        response_metadata={},
                        id='7699923e-eeef-4179-99b5-9e5f62353dec'
                    ),
                    AIMessage(
                        content='您好，我是智能客服【小智】！我先为您查询账户信息~',
                        additional_kwargs={'refusal': None},
                        response_metadata={
                            'token_usage': {
                                'completion_tokens': 149,
                                'prompt_tokens': 613,
                                'total_tokens': 762,
                                'completion_tokens_details': {
                                    'accepted_prediction_tokens': None,
                                    'audio_tokens': None,
                                    'reasoning_tokens': 119,
                                    'rejected_prediction_tokens': None,
                                    'text_tokens': None
                                },
                                'prompt_tokens_details': {
                                    'audio_tokens': None,
                                    'cache_write_tokens': None,
                                    'cached_tokens': 0,
                                    'image_tokens': None,
                                    'text_tokens': None
                                }
                            },
                            'model_provider': 'openai',
                            'model_name': 'glm-5-3-flash-260828',
                            'system_fingerprint': None,
                            'id': '021790523765974923718d11d313173fa258a554449eb87273a3c',
                            'service_tier': 'default',
                            'finish_reason': 'tool_calls',
                            'logprobs': None
                        },
                        id='lc_run--01a0e388-74b9-7351-96cd-1fc1ab298e0c-0',
                        tool_calls=[
                            {
                                'name': 'get_user_by_name',
                                'args': {'name': '张三'},
                                'id': 'call_ro4xglizn6bblv8shb5fg0mj',
                                'type': 'tool_call'
                            }
                        ],
                        invalid_tool_calls=[],
                        usage_metadata={
                            'input_tokens': 613,
                            'output_tokens': 149,
                            'total_tokens': 762,
                            'input_token_details': {'cache_read': 0},
                            'output_token_details': {'reasoning': 119}
                        }
                    )
                ]
            }
        },
        'triggers': ('__pregel_push',)
    }
}

...

--------------------------------------------------

{
    'step': 7,
    'timestamp': '2026-09-27T15:42:56.805849+00:00',
    'type': 'task',
    'payload': {
        'id': '810158ae-e5d3-f51a-4b14-548f8e73ce4d',
        'name': 'model',
        'input': {
            'messages': [
                HumanMessage(
                    content='我叫张三，我想知道我最近买了什么？',
                    additional_kwargs={},
                    response_metadata={},
                    id='7699923e-eeef-4179-99b5-9e5f62353dec'
                ),
                AIMessage(
                    content='您好，我是智能客服【小智】！我先为您查询账户信息~',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 149,
                            'prompt_tokens': 613,
                            'total_tokens': 762,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 119,
                                'rejected_prediction_tokens': None,
                                'text_tokens': None
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'image_tokens': None,
                                'text_tokens': None
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5-3-flash-260828',
                        'system_fingerprint': None,
                        'id': '021790523765974923718d11d313173fa258a554449eb87273a3c',
                        'service_tier': 'default',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0e388-74b9-7351-96cd-1fc1ab298e0c-0',
                    tool_calls=[
                        {
                            'name': 'get_user_by_name',
                            'args': {'name': '张三'},
                            'id': 'call_ro4xglizn6bblv8shb5fg0mj',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 613,
                        'output_tokens': 149,
                        'total_tokens': 762,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 119}
                    }
                ),
                ToolMessage(
                    content="name='张三' email='zhangsan@example.com' phone='138-0000-1234' 
order_no='ORD-2024-88888' order=None",
                    name='get_user_by_name',
                    id='1dc56c19-55a5-4ed9-bc43-7c6b1c1a4999',
                    tool_call_id='call_ro4xglizn6bblv8shb5fg0mj'
                ),
                AIMessage(
                    content='已找到您的账户信息，现在为您查询关联订单的详情~',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 64,
                            'prompt_tokens': 684,
                            'total_tokens': 748,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 30,
                                'rejected_prediction_tokens': None,
                                'text_tokens': None
                            },
                            'prompt_tokens_detai

--------------------------------------------------

{
    'step': 7,
    'timestamp': '2026-09-27T15:43:00.205795+00:00',
    'type': 'task_result',
    'payload': {
        'id': '810158ae-e5d3-f51a-4b14-548f8e73ce4d',
        'name': 'model',
        'error': None,
        'result': {
            'messages': [
                AIMessage(
                    content='工具解析出现异常，我先提交基础账户信息，订单详情稍后用文字为您展示：',
                    additional_kwargs={'refusal': None},
                    response_metadata={
                        'token_usage': {
                            'completion_tokens': 143,
                            'prompt_tokens': 1163,
                            'total_tokens': 1306,
                            'completion_tokens_details': {
                                'accepted_prediction_tokens': None,
                                'audio_tokens': None,
                                'reasoning_tokens': 77,
                                'rejected_prediction_tokens': None,
                                'text_tokens': None
                            },
                            'prompt_tokens_details': {
                                'audio_tokens': None,
                                'cache_write_tokens': None,
                                'cached_tokens': 0,
                                'image_tokens': None,
                                'text_tokens': None
                            }
                        },
                        'model_provider': 'openai',
                        'model_name': 'glm-5-3-flash-260828',
                        'system_fingerprint': None,
                        'id': '021790523776811923718d11d313173fa258a554449eb87e68cf7',
                        'service_tier': 'default',
                        'finish_reason': 'tool_calls',
                        'logprobs': None
                    },
                    id='lc_run--01a0e388-9f46-76a2-995f-bef74e3e73b7-0',
                    tool_calls=[
                        {
                            'name': 'UserInfo',
                            'args': {
                                'email': 'zhangsan@example.com',
                                'name': '张三',
                                'order_no': 'ORD-2024-88888',
                                'phone': '138-0000-1234'
                            },
                            'id': 'call_msnbkxz43li39pusnjew124t',
                            'type': 'tool_call'
                        }
                    ],
                    invalid_tool_calls=[],
                    usage_metadata={
                        'input_tokens': 1163,
                        'output_tokens': 143,
                        'total_tokens': 1306,
                        'input_token_details': {'cache_read': 0},
                        'output_token_details': {'reasoning': 77}
                    }
                ),
                ToolMessage(
                    content="Returning structured response: name='张三' email='zhangsan@example.com' 
phone='138-0000-1234' order_no='ORD-2024-88888' order=None",
                    name='UserInfo',
                    id='98565bc7-a9c5-4af0-a144-e1f7831eea2e',
                    tool_call_id='call_msnbkxz43li39pusnjew124t'
                )
            ],
            'structured_response': UserInfo(
                name='张三',
                email='zhangsan@example.com',
                phone='138-0000-1234',
                order_no='ORD-2024-88888',
                order=None
            )
        },
        'interrupts': []
    }
}

--------------------------------------------------

stream_mode="debug" 是 LangGraph 里信息最全的流式模式，专门用来做深度排查。它不像 values 只给你结果状态，而是把执行过程中的所有内部事件都吐出来，包括 checkpoint 快照、任务调度、结果回填等

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

# 1. 创建 checkpointer 时，直接传给 agent
checkpointer = InMemorySaver()


agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="" 
    "你是智能客服【小智】，专门回答用户关于个人账户的问题" 
    "当用户查询个人账户时" 
    "先查询用户名对应的账户信息是否存在" 
    "再依据账户下的订单号查询对应的账户信息" 
    "依据结果生成格式化用户信息对象"),
    response_format=ToolStrategy(schema=UserInfo),
    checkpointer=checkpointer,   # ← 关键：传进去
    tools=[get_user_by_name,get_order_by_no]
)

# 2. 调用时用 thread_id 标识会话
result = agent.invoke(
    {"messages": ["我叫张三，我想知道我最近买了什么？"]},
    config={
        "configurable": {"thread_id": "session-1"},
        "recursion_limit": 10
    }
)



checkpointer 是 LangGraph 的短期记忆机制，负责把 Agent 在单个会话线程（thread）内的执行状态保存为检查点（checkpoint），实现多轮对话的上下文延续和故障恢复。

In [ ]:
import os
from typing import Literal, Union
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer   # ← custom 模式核心
from rich import print as rprint

load_dotenv(override=True)

model = ChatOpenAI(
    model=os.getenv("ARK_MODEL_NAME"),
    api_key=os.getenv("ARK_API_KEY"),
    base_url=os.getenv("ARK_BASE_URL"),
)

# ---------------- Schema ----------------
class UserInfo(BaseModel):
    """用户账户与订单信息"""
    name: str = Field(description="用户姓名")
    orders: list[str] = Field(default_factory=list, description="订单号列表")
    summary: str = Field(default="", description="订单摘要")

# ---------------- 工具（内部用 writer 上报进度） ----------------
@tool
def get_user_by_name(name: str) -> dict:
    """根据用户名查询账户信息，返回用户是否存在及订单号列表"""
    writer = get_stream_writer()          # ← 获取 writer
    writer({"type": "progress", "stage": "query_user",
            "message": f"正在查询用户：{name}"})

    # 模拟查询
    result = {"exists": True, "name": name, "order_nos": ["ORD-001", "ORD-002"]}

    writer({"type": "progress", "stage": "query_user_done",
            "message": f"用户 {name} 查询完成，共 {len(result['order_nos'])} 个订单"})
    return result

@tool
def get_order_by_no(order_no: str) -> str:
    """根据订单号查询订单详情"""
    writer = get_stream_writer()
    writer({"type": "progress", "stage": "query_order",
            "message": f"正在查询订单：{order_no}"})

    # 模拟查询
    detail = f"订单 {order_no}：已发货，金额 ¥199"

    writer({"type": "progress", "stage": "query_order_done",
            "message": f"订单 {order_no} 查询完成"})
    return detail

# ---------------- Agent ----------------
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content=(
        "你是智能客服【小智】，专门回答用户关于个人账户的问题。"
        "当用户查询个人账户时，先调用 get_user_by_name 查询用户名对应的账户是否存在，"
        "拿到该用户的订单号列表后，再对每个订单号调用 get_order_by_no 查询订单详情，"
        "最后依据结果生成格式化的用户信息对象。"
    )),
    tools=[get_user_by_name, get_order_by_no],
    response_format=ToolStrategy(schema=UserInfo),
    checkpointer=checkpointer,
)

# ---------------- 用 custom 模式消费进度 ----------------
for mode, chunk in agent.stream(
    {"messages": ["我叫张三，我想知道我最近买了什么？"]},
    stream_mode=["updates", "custom"],     # ← 同时看进度和节点更新
    config={
        "configurable": {"thread_id": "session-1"},
        "recursion_limit": 10,
    },
):
    if mode == "custom":
        # chunk 就是你 writer 发出的原始 dict
        rprint(f"[进度] {chunk.get('message')}")
    elif mode == "updates":
        for node, update in chunk.items():
            rprint(f"[节点 {node}] {update}")

custom 输出模式（自定义输出模式）用于让开发者在工具或节点内部主动发送自定义数据，用于输出业务逻辑相关的进度信息、自定义日志或指标。

适用场景
自定义输出模式适合长耗时任务的进度反馈。比如数据处理流程中，每处理完一条记录就发送一次进度更新，让客户端能实时看到处理进度，而不是等到整个节点执行完毕才知道结果。自定义数据通常包含 type 字段用于客户端路由，以及 current/total 等进度数据用于渲染进度条。